# Workshop · Fine-tuning con 🤗 Trainer
## Clasificación de reseñas de Yelp

---

En este taller vas a construir un clasificador de sentimiento sobre reseñas de Yelp usando la API **Trainer** de HuggingFace Transformers.

### Etapas

| Etapa | Contenido |
|-------|-----------|
| **1 — Preparación de datos** | Cargar CSV · Exploración · Mapeo de etiquetas · Tokenización |
| **2 — Fine-tuning y evaluación** | Modelo · Métricas · TrainingArguments · Trainer · Análisis |

**Modelo base:** `distilbert-base-uncased` (~66M params, rápido en una GPU)  
**Dataset:** Yelp Review Full exportado a CSV (entregado por el profesor)

> **Instrucciones:** Completa cada celda marcada con `# YOUR CODE HERE`. Las celdas de código sin TODO ya están implementadas — ejecútalas directamente.

In [ ]:
# Ejecuta esta celda una sola vez y reinicia el kernel antes de continuar
%pip install -q "transformers>=5.0" datasets evaluate accelerate scikit-learn seaborn

## 0. Imports

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch  : 2.8.0+cu128
Device   : cuda


## ETAPA 1 — PREPARACIÓN DE DATOS

### 1.1 Cargar los datos

El profesor entrega dos archivos CSV:

- `yelp_train.csv` — columnas: `text`, `stars`  
- `yelp_test.csv`  — columnas: `text`, `stars`

`stars` es un entero entre 1 y 5.

In [ ]:
!git clone https://huggingface.co/datasets/Yelp/yelp_review_full

In [ ]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
train_df = pd.read_parquet(splits["train"])
test_df  = pd.read_parquet(splits["test"])

print(f"Train: {len(train_df):,} filas")
print(f"Test : {len(test_df):,} filas")
print(train_df.head())

### 1.2 Exploración

#### 📝 TODO 1
Visualiza la distribución de estrellas en el split de train.
Usa un countplot o barplot con seaborn / matplotlib.
¿Está balanceado el dataset?

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE


#### 📝 TODO 2
Imprime 3 ejemplos de cada valor de stars (1 a 5).
¿Puedes identificar patrones de lenguaje en cada categoría?

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE


#### 📝 TODO 3
Calcula y visualiza la distribución de longitudes de texto (en palabras).
¿Cuál es la longitud media? ¿Hay reseñas muy largas que podrían truncarse?

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE


### 1.3 Mapeo de etiquetas

Las 5 estrellas de Yelp se colapsan en 3 clases de sentimiento:

| Estrellas | Clase | Label |
|-----------|-------|-------|
| ⭐ 1-2    | negative | 0 |
| ⭐⭐⭐ 3  | neutral  | 1 |
| ⭐⭐⭐⭐ 4-5 | positive | 2 |

Este mapeo reduce el problema y alinea con benchmarks estándar de análisis de sentimiento.

In [ ]:
LABEL_NAMES = ["negative", "neutral", "positive"]
NUM_CLASSES  = len(LABEL_NAMES)

def map_stars_to_label(stars: int) -> int:
    if stars <= 2:
        return 0   # negative
    elif stars == 3:
        return 1   # neutral
    else:
        return 2   # positive

train_df["label"] = train_df["stars"].apply(map_stars_to_label)
test_df["label"]  = test_df["stars"].apply(map_stars_to_label)

print("\nDistribución de clases (train):")
print(train_df["label"].value_counts().sort_index()
      .rename(index={i: name for i, name in enumerate(LABEL_NAMES)}))

#### 📝 TODO 4
Visualiza la distribución de las 3 clases resultantes.
¿Sigue estando desbalanceado? ¿Qué impacto puede tener esto en el modelo?

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE


### 1.4 Preprocesamiento de texto

#### 📝 TODO 5
Implementa una función clean_review(text) que normalice el texto.
Como mínimo considera:
  - Eliminar URLs
  - Normalizar espacios
Puedes agregar más pasos si lo consideras útil.
Aplícala a las columnas "text" de ambos DataFrames.

def clean_review(text: str) -> str:
    # YOUR CODE HERE
    pass

train_df["text"] = ... \
test_df["text"]  = ...

In [ ]:
# YOUR CODE HERE


### 1.5 Convertir a HuggingFace Dataset

La API Trainer trabaja con objetos `Dataset` de HuggingFace, no con DataFrames de pandas.  
`Dataset.from_pandas()` hace la conversión directamente.  
`DatasetDict` agrupa los splits en un solo objeto.

#### 📝 TODO 6
Crea un DatasetDict con splits "train" y "test" a partir de los DataFrames.
Cada Dataset debe tener solo las columnas "text" y "label".

Pistas:
  Dataset.from_pandas(df[["text", "label"]])
  DatasetDict({"train": ..., "test": ...})

raw_datasets = ...
print(raw_datasets)

In [ ]:
# YOUR CODE HERE


### 1.6 Tokenización

#### 📝 TODO 7
Carga el tokenizer de "distilbert-base-uncased" con AutoTokenizer.
Imprime:
  - El tipo de tokenizer
  - El tamaño del vocabulario
  - Un ejemplo de tokenización sobre una reseña del dataset

tokenizer = ...

In [ ]:
# YOUR CODE HERE


#### 📝 TODO 8
Implementa tokenize_function(batch) y aplícala sobre raw_datasets con .map().

Requisitos:
  - Truncar a MAX_LEN = 128
  - NO hacer padding aquí (lo hará DataCollatorWithPadding por batch)
  - Eliminar la columna "text" después de tokenizar
  - Renombrar "label" → "labels" (convención del Trainer)

Pistas:
  tokenizer(batch["text"], truncation=True, max_length=128)
  dataset.map(fn, batched=True, remove_columns=["text"])
  dataset.rename_column("label", "labels")

tokenized_datasets = ...
print(tokenized_datasets)

In [ ]:
# YOUR CODE HERE


#### 📝 TODO 9
Crea el DataCollatorWithPadding.
¿Por qué es preferible el padding dinámico al padding estático a MAX_LEN?

data_collator = ...

In [ ]:
# YOUR CODE HERE


## ETAPA 2 — FINE-TUNING Y EVALUACIÓN

### 2.1 Modelo

#### 📝 TODO 10
Carga el modelo con AutoModelForSequenceClassification.
Configura:
  - num_labels = NUM_CLASSES
  - id2label   = {0: "negative", 1: "neutral", 2: "positive"}
  - label2id   = {"negative": 0, "neutral": 1, "positive": 2}
  - ignore_mismatched_sizes = True

Imprime el número de parámetros totales y entrenables.

model = ...

In [ ]:
# YOUR CODE HERE


### 2.2 Métricas

#### 📝 TODO 11
Implementa compute_metrics(eval_pred).

eval_pred es un EvalPrediction con:
  .predictions : logits  (N, num_classes)  — salida cruda del modelo
  .label_ids   : labels  (N,)              — etiquetas reales

La función debe retornar un dict con al menos:
  - "accuracy"
  - "f1_macro"

Usa evaluate.load("accuracy") y evaluate.load("f1").

def compute_metrics(eval_pred):
    # YOUR CODE HERE
    pass

In [ ]:
# YOUR CODE HERE


### 2.3 TrainingArguments

#### 📝 TODO 12
Configura TrainingArguments. Como mínimo especifica:
  - output_dir
  - num_train_epochs      = 3
  - per_device_train_batch_size = 32
  - per_device_eval_batch_size  = 32
  - learning_rate         = 2e-5
  - weight_decay          = 0.01
  - eval_strategy         = "epoch"
  - save_strategy         = "epoch"
  - load_best_model_at_end = True
  - metric_for_best_model = "f1_macro"
  - fp16 = True  si hay GPU disponible

Pregunta: ¿qué hace warmup_ratio? ¿Por qué es útil en fine-tuning?

training_args = TrainingArguments(...)

In [ ]:
# YOUR CODE HERE


### 2.4 Trainer

#### 📝 TODO 13
Crea el Trainer con todos sus argumentos:
  - model
  - args            (training_args)
  - train_dataset   (split "train" del dataset tokenizado)
  - eval_dataset    (split "test")
  - processing_class (tokenizer)
  - data_collator
  - compute_metrics

trainer = Trainer(...)

In [ ]:
# YOUR CODE HERE


### 2.5 Entrenamiento

#### 📝 TODO 14
Ejecuta el entrenamiento con trainer.train().
Guarda el resultado en train_result e imprime:
  - Pasos totales
  - Train loss final
  - Tiempo total de entrenamiento

train_result = ...

In [ ]:
# YOUR CODE HERE


### 2.6 Evaluación

#### 📝 TODO 15
Evalúa el modelo en el split de test con trainer.predict().
A diferencia de trainer.evaluate(), predict() también devuelve los logits
y etiquetas, lo que permite hacer análisis detallados.

Con los resultados:
  1. Imprime las métricas de test (accuracy y f1_macro)
  2. Genera el classification_report de sklearn con digits=4
  3. Visualiza la matriz de confusión normalizada

Preguntas:
  - ¿Cuál clase tiene peor recall? ¿Por qué crees que ocurre?
  - ¿Qué diferencia hay entre accuracy y f1_macro en este caso?

test_output = trainer.predict(...)

In [ ]:
# YOUR CODE HERE


### 2.7 Análisis de errores

#### 📝 TODO 16
Muestra 5 ejemplos de errores del modelo para cada par (real, predicho).
Por ejemplo: reseñas que eran "positive" pero se clasificaron como "negative".

Para acceder al texto original después de tokenizar puedes usar test_df.
Asegúrate de alinear los índices correctamente.

Pregunta: ¿los errores tienen sentido? ¿Son casos ambiguos?

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE


### 2.8 Guardar el modelo

#### 📝 TODO 17
Guarda el modelo y el tokenizer en un directorio local con:
  trainer.save_model("mi_modelo_yelp")
  tokenizer.save_pretrained("mi_modelo_yelp")

Luego cárgalo de nuevo con from_pretrained() y verifica que las predicciones
sobre 3 reseñas inventadas por ti sean correctas.

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE
